#Classifier des embeddings avec Keras et l'API Gemini

## Présentation

Bienvenue dans ce nouveau notebook du cours sur l'IA générative. Vous allez apprendre à utiliser les embeddings produits par l'API Gemini pour entraîner un modèle capable de classer des messages de forums (newsgroups) dans leur catégorie (le forum d'origine), à partir de leur contenu.

Cette technique utilise les embeddings de l'API Gemini comme entrée, ce qui évite d'entraîner un modèle directement sur du texte. Résultat : elle donne de bons résultats avec relativement peu d'exemples, comparé à l'entraînement d'un modèle de texte à partir de zéro.

> **Version adaptée pour Google Colab** : ce notebook est en français, lit la clé API depuis les **secrets Colab**, relance automatiquement les appels en cas de quota atteint et utilise le modèle `gemini-embedding-001` (le modèle `text-embedding-004` d'origine a été arrêté le 14 janvier 2026). Les embeddings sont calculés **par lots** pour limiter le nombre de requêtes.

## Installer le SDK

In [ ]:
# Colab inclut déjà google-genai : on le met simplement à jour vers la dernière version
# (nécessaire pour les modèles Gemini récents).
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 810.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.0 which is incompatible.


In [ ]:
# Importation du nouveau SDK officiel de Google pour l'API Gemini
from google import genai
from google.genai import types

# Affichage de la version installée pour vérification
print(f"Version du SDK Google GenAI : {genai.__version__}")

# Définition du modèle d'embedding utilisé tout au long de ce projet
EMBEDDING_MODEL = "gemini-embedding-001"

### Configurer votre clé API

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**, saisissez `GOOGLE_API_KEY` comme nom et collez votre clé comme valeur.
3. Activez l'interrupteur **Accès au notebook**.

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

Hors de Colab (exécution locale, VM…), la cellule utilise la variable d'environnement `GOOGLE_API_KEY`.

In [ ]:
import os

GOOGLE_API_KEY = None

# 1) Tentative de récupération de la clé API depuis les secrets de Google Colab
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # En dehors de Google Colab, cette bibliothèque n'est pas disponible

# 2) Alternative : récupération de la clé via les variables d'environnement (exécution locale)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

# Si aucune clé n'est trouvée, on lève une erreur explicative
if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

# Initialisation du client de l'API Gemini avec la clé récupérée
client = genai.Client(api_key=GOOGLE_API_KEY)
print("Clé API chargée avec succès ✅")

Clé API chargée ✅


Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

## Jeu de données

Le jeu de données [20 Newsgroups](https://scikit-learn.org/stable/datasets/real_world.html#newsgroups-dataset) contient 18 000 messages de forums sur 20 sujets, répartis en un ensemble d'entraînement et un ensemble de test. La séparation entre les deux se fait selon la date : messages publiés avant et après une date donnée. Dans ce tutoriel, vous utiliserez des sous-échantillons de ces deux ensembles, avec un peu de traitement dans Pandas.

In [ ]:
# Importation du chargeur de jeu de données depuis Scikit-Learn
from sklearn.datasets import fetch_20newsgroups

# Téléchargement des ensembles de données d'entraînement et de test séparément
newsgroups_train = fetch_20newsgroups(subset="train")
newsgroups_test = fetch_20newsgroups(subset="test")

# Affichage de la liste des 20 catégories de forums disponibles
display(newsgroups_train.target_names)

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

Voici à quoi ressemble un enregistrement de l'ensemble d'entraînement.

In [ ]:
# Affichage d'un exemple concret de document brut (le tout premier du jeu de données)
print("--- Exemple de message brut ---")
print(newsgroups_train.data[0])

From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Thanks,
- IL
   ---- brought to you by your neighborhood Lerxst ----







Commencez par prétraiter les données dans un dataframe Pandas. Pour retirer les informations sensibles comme les noms et les adresses e-mail, vous ne garderez que le sujet et le corps de chaque message. Cette étape facultative transforme les données en texte plus générique (plutôt que des e-mails), pour qu'elles soient utilisables dans d'autres contextes.

In [ ]:
import email
import re
import pandas as pd

def preprocess_newsgroup_row(data):
    """
    Nettoie un document brut en ne conservant que l'objet et le corps,
    et en supprimant les adresses e-mail pour anonymiser.
    """
    # Analyse du format d'email brut pour structurer le texte
    msg = email.message_from_string(data)
    text = f"{msg['Subject']}\n\n{msg.get_payload()}"

    # Expression régulière pour supprimer les adresses e-mail résiduelles
    text = re.sub(r"[\w\.-]+@[\w\.-]+", "", text)

    # Limitation de la taille du texte à 5000 caractères pour respecter les quotas
    text = text[:5000]
    return text

def preprocess_newsgroup_data(newsgroup_dataset):
    """
    Convertit l'ensemble des données d'origine en DataFrame Pandas structuré
    et applique le nettoyage textuel.
    """
    # Création du DataFrame initial
    df = pd.DataFrame(
        {"Text": newsgroup_dataset.data, "Label": newsgroup_dataset.target}
    )

    # Application de notre fonction de nettoyage sur chaque cellule de texte
    df["Text"] = df["Text"].apply(preprocess_newsgroup_row)

    # Association de l'identifiant numérique de la classe à son nom textuel complet
    df["Class Name"] = df["Label"].map(lambda l: newsgroup_dataset.target_names[l])
    return df

Ensuite, vous allez échantillonner les données : un nombre fixe d'exemples par catégorie dans l'ensemble d'entraînement, en ne gardant que quelques catégories pour ce tutoriel. Choisissez les catégories scientifiques (`sci`) à comparer.

In [ ]:
# Application du prétraitement complet sur les deux jeux de données
df_train = preprocess_newsgroup_data(newsgroups_train)
df_test = preprocess_newsgroup_data(newsgroups_test)

# Aperçu des 5 premières lignes traitées de l'ensemble d'entraînement
display(df_train.head())

,Text,Label,Class Name
0,WHAT car is this!?\n\n I was wondering if anyo...,7,rec.autos
1,SI Clock Poll - Final Call\n\nA fair number of...,4,comp.sys.mac.hardware
2,"PB questions...\n\nwell folks, my mac plus fin...",4,comp.sys.mac.hardware
3,Re: Weitek P9000 ?\n\nRobert J.C. Kyanko () wr...,1,comp.graphics
4,Re: Shuttle Launch Question\n\nFrom article <>...,14,sci.space


In [ ]:
def sample_data(df, num_samples, classes_to_keep):
    """
    Échantillonne de manière équilibrée un nombre défini de messages par classe
    et filtre uniquement sur les catégories cibles (par exemple celles contenant 'sci').
    """
    # Sélection d'un nombre égal d'échantillons aléatoires pour chaque classe
    df = (
        df.groupby("Label")[df.columns]
        .apply(lambda x: x.sample(num_samples))
        .reset_index(drop=True)
    )

    # Conservation uniquement des lignes correspondant au filtre textuel de catégorie
    df = df[df["Class Name"].str.contains(classes_to_keep)]

    # Comme nous avons restreint les catégories, nous recalculons des étiquettes numériques propres (0, 1, 2, ...)
    df["Class Name"] = df["Class Name"].astype("category")
    df["Encoded Label"] = df["Class Name"].cat.codes
    return df

In [ ]:
# Configuration de l'échantillonnage pour éviter de saturer les quotas gratuits de l'API
TRAIN_NUM_SAMPLES = 50  # 50 exemples d'entraînement par catégorie conservée
TEST_NUM_SAMPLES = 10   # 10 exemples de validation par catégorie conservée

# Filtre pour cibler uniquement les forums scientifiques (ex: sci.space, sci.med...)
CLASSES_TO_KEEP = "sci"

# Application de l'échantillonnage sur nos données nettoyées
df_train = sample_data(df_train, TRAIN_NUM_SAMPLES, CLASSES_TO_KEEP)
df_test = sample_data(df_test, TEST_NUM_SAMPLES, CLASSES_TO_KEEP)

In [ ]:
# Validation visuelle de la distribution équilibrée des classes pour l'entraînement
print("Nombre de messages d'entraînement par classe scientifique :")
display(df_train.value_counts("Class Name"))

,count
Class Name,
sci.crypt,50
sci.electronics,50
sci.med,50
sci.space,50


In [ ]:
# Validation visuelle de la distribution équilibrée des classes pour le test
print("Nombre de messages de test par classe scientifique :")
display(df_test.value_counts("Class Name"))

,count
Class Name,
sci.crypt,10
sci.electronics,10
sci.med,10
sci.space,10


## Créer les embeddings

Dans cette section, vous allez générer un embedding pour chaque texte avec le point d'accès d'embeddings de l'API Gemini. Pour en savoir plus, consultez le [guide des embeddings](https://ai.google.dev/gemini-api/docs/embeddings).

**Remarque** : dans le notebook d'origine, les embeddings étaient calculés un par un. Ici, ils sont envoyés **par lots de 50 textes** : on fait donc beaucoup moins de requêtes, mais le nombre de tokens reste le même. Avec une clé gratuite, cette étape peut prendre quelques minutes si le quota de tokens par minute est atteint : le retry attend alors automatiquement.

### Types de tâche

Le modèle `gemini-embedding-001` accepte un paramètre de type de tâche (`task_type`), qui produit des embeddings adaptés à la tâche visée.

Type de tâche | Description
---       | ---
RETRIEVAL_QUERY	| Le texte est une requête dans un contexte de recherche/récupération.
RETRIEVAL_DOCUMENT | Le texte est un document dans un contexte de recherche/récupération.
SEMANTIC_SIMILARITY	| Le texte servira à mesurer la similarité sémantique (STS).
CLASSIFICATION	| Les embeddings serviront à de la classification.
CLUSTERING	| Les embeddings serviront à du regroupement (clustering).
FACT_VERIFICATION | Le texte servira à vérifier des faits.

Dans cet exemple, vous allez faire de la classification.

La politique de retry est un peu plus patiente que dans les autres notebooks (jusqu'à 5 minutes), car le quota de tokens par minute se réinitialise au bout d'une minute.

In [ ]:
from google.api_core import retry
import tqdm
from tqdm.rich import tqdm as tqdmr
import warnings

# Intégration de la barre de progression tqdm dans l'écosystème Pandas
tqdmr.pandas()

# Désactivation des messages d'avertissement liés aux fonctionnalités expérimentales de tqdm
warnings.filterwarnings("ignore", category=tqdm.TqdmExperimentalWarning)

# Définition de la stratégie de relance automatique (retry) face aux limites de l'API (codes 429 et 503)
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})
politique_retry = retry.Retry(predicate=is_retriable, initial=2, maximum=30, timeout=300)

TAILLE_LOT = 50  # Nombre de textes envoyés en un seul appel API groupé (batching)

@politique_retry
def embed_batch(texts: list[str]) -> list[list[float]]:
    """
    Appelle l'API Gemini pour obtenir l'embedding vectoriel d'une liste de textes.
    L'utilisation du type de tâche 'classification' optimise les vecteurs générés.
    """
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type="classification",
        ),
    )
    # Extraction de la liste de valeurs numériques (les coordonnées du vecteur d'embedding)
    return [e.values for e in response.embeddings]

def embed_fn(text: str) -> list[float]:
    """
    Fonction utilitaire pour calculer le vecteur d'un texte unique.
    """
    return embed_batch([text])[0]

def create_embeddings(df):
    """
    Parcourt le DataFrame et génère les vecteurs d'embeddings par blocs pour accélérer le traitement.
    """
    textes = df["Text"].tolist()
    embeddings = []
    # Découpage et envoi par paquets (lots de 50) pour limiter la surcharge de requêtes
    for debut in tqdm.tqdm(range(0, len(textes), TAILLE_LOT), desc="Lots"):
        embeddings.extend(embed_batch(textes[debut:debut + TAILLE_LOT]))
    df["Embeddings"] = embeddings
    return df

Ce code privilégie la lisibilité. Pour aller plus loin, vous pouvez essayer une génération d'embeddings parallèle ou asynchrone (`client.aio`), ou l'[API Batch](https://ai.google.dev/gemini-api/docs/batch-api), moins chère mais asynchrone. Cette étape prend un peu de temps.

In [ ]:
# Génération et intégration des embeddings vectoriels pour nos deux ensembles de données
print("--- Génération des embeddings d'entraînement ---")
df_train = create_embeddings(df_train)

print("\n--- Génération des embeddings de test ---")
df_test = create_embeddings(df_test)

Lots: 100%|██████████| 1/1 [01:01<00:00, 61.71s/it]


## Construire un modèle de classification

Vous allez définir un modèle simple qui prend en entrée les embeddings bruts, avec une couche cachée et une couche de sortie qui donne les probabilités de chaque classe. La prédiction correspond à la probabilité qu'un texte appartienne à une catégorie donnée.

Pendant l'exécution, Keras s'occupe des détails comme le mélange des données, le calcul des métriques et le reste du code habituel de ML.

In [ ]:
# Aperçu de la structure finale contenant les vecteurs générés pour l'entraînement
display(df_train.head())

,Text,Label,Class Name,Encoded Label,Embeddings
550,"Re: Once tapped, your code is no good any more...",11,sci.crypt,0,"[-0.009862087, -0.0055224383, 0.026557328, -0...."
551,Re: Tempest\n\n\n (Rob Stampfli) writes:\n\n> ...,11,sci.crypt,0,"[-0.00051401765, -0.0049106013, 0.011601175, -..."
552,Re: Why the clipper algorithm is secret\n\nIn ...,11,sci.crypt,0,"[-0.014383721, -0.0084365085, 0.0031569335, -0..."
553,"Re: Once tapped, your code is no good any more...",11,sci.crypt,0,"[-0.012094427, 0.014304722, 0.026739534, -0.05..."
554,"Re: Once tapped, your code is no good any more...",11,sci.crypt,0,"[-0.01279728, -0.011016436, 0.022230368, -0.05..."


In [ ]:
# Aperçu de la structure finale contenant les vecteurs générés pour le test
display(df_test.head())

,Text,Label,Class Name,Encoded Label,Embeddings
110,Re: Audio CDs?\n\n>>>>> On 30 Apr 1993 21:16:2...,11,sci.crypt,0,"[-0.0127824955, -0.013707176, 0.0064001367, -0..."
111,Citations for Electronic Cash papers please\n\...,11,sci.crypt,0,"[-0.032866202, -0.024862926, 0.0020574236, -0...."
112,clipper serial numbers need 2nd court order?\n...,11,sci.crypt,0,"[-0.028527858, -0.015025751, 0.005132659, -0.0..."
113,Pseudocollisions in MD5\n\nFollowing is a shor...,11,sci.crypt,0,"[-0.0039835493, 0.0020292955, -0.0137959365, -..."
114,Re: PGP Where to get it?\n\n>At the moment PGP...,11,sci.crypt,0,"[-0.0147400675, 0.00068252085, 0.0033699272, -..."


In [ ]:
import keras
from keras import layers

def build_classification_model(input_size: int, num_classes: int) -> keras.Model:
    """
    Construit un réseau de neurones Keras simple :
    - Une couche d'entrée de taille égale à la dimension de l'embedding.
    - Une couche dense intermédiaire (relu) pour capturer les relations.
    - Une couche de sortie softmax donnant une probabilité pour chaque catégorie.
    """
    return keras.Sequential(
        [
            layers.Input([input_size], name="embedding_inputs"),
            layers.Dense(input_size, activation="relu", name="hidden"),
            layers.Dense(num_classes, activation="softmax", name="output_probs"),
        ]
    )

In [ ]:
# Récupération automatique de la taille du vecteur d'embedding (3072)
embedding_size = len(df_train["Embeddings"].iloc[0])

# Détermination du nombre unique de classes scientifiques ciblées
num_classes = len(df_train["Class Name"].unique())

# Initialisation du modèle
classifier = build_classification_model(embedding_size, num_classes)

# Affichage de l'architecture détaillée du réseau de neurones
classifier.summary()

# Compilation avec la fonction de perte Sparse Categorical Crossentropy (adaptée aux labels encodés en entiers)
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden (Dense)                  │ (None, 3072)           │     9,440,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_probs (Dense)            │ (None, 4)              │        12,292 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,452,548 (36.06 MB)

 Trainable params: 9,452,548 (36.06 MB)

 Non-trainable params: 0 (0.00 B)

## Entraîner le modèle

Vous pouvez enfin entraîner votre modèle. Ce code utilise l'arrêt anticipé (*early stopping*) pour sortir de la boucle d'entraînement dès que la précision se stabilise : le nombre d'époques réellement exécutées peut donc être inférieur à la valeur indiquée.

In [ ]:
import numpy as np

NUM_EPOCHS = 20
BATCH_SIZE = 32

# Extraction des étiquettes cibles sous forme d'entiers
y_train = df_train["Encoded Label"]
y_val = df_test["Encoded Label"]

# Conversion des listes d'embeddings en tableaux NumPy 2D pour Keras
x_train = np.stack(df_train["Embeddings"])
x_val = np.stack(df_test["Embeddings"])

# Configuration de l'arrêt anticipé si la précision d'entraînement stagne pendant 3 époques
early_stop = keras.callbacks.EarlyStopping(monitor="accuracy", patience=3)

# Entraînement du classifieur de réseau de neurones
history = classifier.fit(
    x=x_train,
    y=y_train,
    validation_data=(x_val, y_val),
    callbacks=[early_stop],
    batch_size=BATCH_SIZE,
    epochs=NUM_EPOCHS,
)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 219ms/step - accuracy: 1.0000 - loss: 0.0075 - val_accuracy: 0.9500 - val_loss: 0.2061
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 268ms/step - accuracy: 1.0000 - loss: 0.0062 - val_accuracy: 0.9500 - val_loss: 0.2066
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 173ms/step - accuracy: 1.0000 - loss: 0.0053 - val_accuracy: 0.9500 - val_loss: 0.2171
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 280ms/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 0.9500 - val_loss: 0.2165


## Évaluer les performances du modèle

Utilisez <a href="https://keras.io/api/models/model_training_apis/#evaluate-method"><code>Model.evaluate</code></a> de Keras pour calculer la perte et la précision sur l'ensemble de test.

In [ ]:
# Évaluation finale du modèle entraîné sur l'ensemble de validation
evals = classifier.evaluate(x=x_val, y=y_val, return_dict=True)
print(f"\nPrécision de validation (Accuracy) : {evals['accuracy'] * 100:.2f}%")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9500 - loss: 0.2007


{'accuracy': 0.949999988079071, 'loss': 0.20068156719207764}

Pour en savoir plus sur l'entraînement de modèles avec Keras, notamment la visualisation des métriques d'entraînement, lisez [Training & evaluation with built-in methods](https://keras.io/guides/training_with_built_in_methods/).

## Essayer une prédiction personnalisée

Maintenant que vous avez un modèle entraîné avec de bonnes métriques, essayez une prédiction sur de nouvelles données écrites à la main. Utilisez l'exemple fourni, ou vos propres données, pour voir comment le modèle se comporte.

In [ ]:
def make_prediction(text: str) -> list[float]:
    """
    Fonction globale pour classer un nouveau texte :
    1. Calcule l'embedding de ce texte via l'API.
    2. Passe l'embedding dans le classifieur Keras.
    3. Retourne le vecteur de probabilités de sortie.
    """
    # 1. Calcul vectoriel
    embedded = embed_fn(text)

    # 2. Adaptation du format de l'entrée (Keras attend toujours un batch, même de taille 1)
    inp = np.array([embedded])

    # 3. Prédiction et désencapsulage du résultat unique
    [result] = classifier.predict(inp)
    return result

L'exemple ci-dessous est en anglais, comme les données d'entraînement. En substance : « Débutant qui cherche à partir d'ici. Je voudrais voyager jusqu'aux confins ! Quel engin puis-je acheter ? Quel est le plus facile d'accès depuis ce 3e rocher ? »

In [ ]:
# Exemple de texte sans vocabulaire direct mais sémantiquement lié à l'espace ('craft', '3rd rock', 'travelling')
new_text = """
First-timer looking to get out of here.

Hi, I'm writing about my interest in travelling to the outer limits!

What kind of craft can I buy? What is easiest to access from this 3rd rock?

Let me know how to do that please.
"""

# Calcul des probabilités de classes pour notre exemple de texte
result = make_prediction(new_text)

# Affichage des prédictions détaillées pour chaque catégorie
for idx, category in enumerate(df_test["Class Name"].cat.categories):
    print(f"{category}: {result[idx] * 100:0.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
sci.crypt: 0.00%
sci.electronics: 0.15%
sci.med: 0.15%
sci.space: 99.70%
